# NILM – Semaine 3 : un réseau de neurones contre le CO (variante « optimisée »)

**Objectif de la semaine.** Entraîner un **réseau de neurones convolutif** qui lit la courbe de la maison entière et prédit, minute par minute, la puissance et l'état de chaque appareil, puis le comparer au **CO** et au **naïf** sur **exactement le même test**.

**Ce que fait le réseau.** Il reçoit la courbe totale sur une longue fenêtre (plusieurs heures de contexte) et en tire, pour chaque appareil : (1) une puissance, (2) une probabilité d'être allumé. Là où le CO ne voit que la valeur de la minute, le réseau voit la *forme* de la courbe avant et après.

**Ce notebook** contient la **variante élargie** que j'ai testée après la première version du réseau : deux fois plus de canaux (64), un bloc de dilatation supplémentaire (jusqu'à 128 minutes), des blocs *Squeeze-and-Excitation*, du dropout, une *focal loss*, l'optimiseur AdamW, un pas d'apprentissage adaptatif et un tirage des morceaux d'entraînement favorisant les appareils allumés. La version de base (32 canaux, 22 120 paramètres) est reprise à l'identique dans les notebooks des semaines 4 à 6.

**Données.** UK-DALE maison 1 sur un an (mars 2013 à février 2014), moyennée à la minute. Le notebook a été exécuté sous Colab avec un GPU T4 (il fonctionne aussi sur CPU, plus lentement).

**Règle de décision du projet :** je ne garde le réseau que s'il bat le CO sur ce test.

## 1. Réglages
- données et appareils : identiques à la semaine 2, mais sur **un an** (`LOAD`) ;
- découpe : 30 % de la fin pour le test (`TEST_FRAC`), les 7 % juste avant pour la validation (`VAL_FRAC`), le reste pour l'entraînement ;
- `USE_TIME` : ajoute l'heure de la journée aux entrées du réseau ;
- entraînement : morceaux de 12 h (`CROP` = 720 min), lots de 32, 100 pas par époque, 50 époques au plus, arrêt après 12 époques sans progrès ;
- réseau : `CH` = 64 canaux internes ;
- régularisation et optimisation : `WEIGHT_DECAY` (AdamW), `USE_FOCAL` et `FOCAL_GAMMA` (focal loss), `GRAD_CLIP` (écrêtage du gradient), `SCHEDULER_PATIENCE` et `SCHEDULER_FACTOR` (le pas d'apprentissage est divisé par 2 quand la validation stagne).

In [ ]:
from pathlib import Path
from itertools import product
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# ================== RÉGLAGES (à adapter) ==================
DATA_DIR = Path("/content")
MAINS = "aggregate"
APPLIANCES = ["fridge", "kettle", "washing_machine", "dishwasher"]
LOAD = ("2013-03-01", "2014-02-28")          # fenêtre de dates lue dans les fichiers
RESAMPLE = "1min"
ON_W = {"fridge": 30, "kettle": 500, "washing_machine": 20, "dishwasher": 20}    # seuil « allumé » (W)
N_ON_STATES = {"fridge": 1, "kettle": 1, "washing_machine": 3, "dishwasher": 3}  # niveaux du CO
TEST_FRAC, VAL_FRAC = 0.30, 0.07             # fin des données = test ; juste avant = validation
USE_TIME = True                              # ajouter l'heure de la journée comme entrée du réseau
CROP, BATCH, STEPS, EPOCHS, PATIENCE = 720, 32, 100, 50, 12   # morceaux de 12 h ; 100 pas par époque
CH, LR = 64, 1e-3                            # largeur du réseau (doublée), pas d'apprentissage
WEIGHT_DECAY = 1e-4                          # régularisation AdamW
USE_FOCAL = True                             # Focal Loss pour le déséquilibre ON/OFF
FOCAL_GAMMA = 2.0                            # paramètre focal loss
GRAD_CLIP = 1.0                              # gradient clipping
SCHEDULER_PATIENCE = 5                       # patience du scheduler LR
SCHEDULER_FACTOR = 0.5                       # facteur de réduction LR
# ===========================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("calcul sur :", device, "(GPU conseillé)" if device == "cpu" else "")
torch.manual_seed(0); np.random.seed(0)
plt.rcParams["figure.figsize"] = (11, 3.2)
plt.rcParams["axes.grid"] = True

## 2. Chargement, grille de temps complète et découpe
Je lis les fichiers UK-DALE de la maison 1 sur un an et je ramène chaque canal à la minute. Je recale tous les canaux sur **une grille de temps complète** (une ligne par minute, même quand une mesure manque), indispensable pour un réseau qui lit des fenêtres continues. Une minute n'est **valide** que si tous les canaux ont une mesure ; les minutes invalides sont mises à zéro en entrée et exclues de la perte et des scores.

La découpe est chronologique : entraînement du 1er mars au 16 octobre 2013, validation jusqu'au 11 novembre 2013, test du 11 novembre 2013 au 28 février 2014 (≈ 152 700 minutes valides). Sur l'année, 95,2 % des minutes sont valides.

In [ ]:
def read_labels(folder):
    labels = {}
    for line in (folder / "labels.dat").read_text().splitlines():
        parts = line.split(maxsplit=1)
        if len(parts) == 2:
            labels[parts[1].strip()] = int(parts[0])
    return labels

def read_channel(folder, ch):
    df = pd.read_csv(folder / f"channel_{ch}.dat", sep=r"\s+", header=None, usecols=[0, 1], names=["t", "w"])
    df.index = pd.to_datetime(df["t"], unit="s")
    s = df["w"].astype(float)
    s = s[~s.index.duplicated()].sort_index()
    return s.loc[LOAD[0]:LOAD[1]]

labels = read_labels(DATA_DIR)
series = {}
for name in [MAINS] + APPLIANCES:
    if name not in labels:
        print("absent de labels.dat :", name); continue
    try:
        s = read_channel(DATA_DIR, labels[name])
    except FileNotFoundError:
        print(f"fichier manquant : channel_{labels[name]}.dat ({name})"); continue
    if s.empty:
        print("aucune donnée sur la période pour", name); continue
    series[name] = s.resample(RESAMPLE).mean()
    print(f"chargé : {name:16s} {len(s):>8} points  {s.index.min()} -> {s.index.max()}")
if MAINS not in series:
    raise SystemExit("Canal maison entière introuvable : vérifie les fichiers envoyés dans /content.")
APPLIANCES = [a for a in APPLIANCES if a in series]
n_app = len(APPLIANCES)

data = pd.DataFrame(series)
grid = pd.date_range(data.index.min(), data.index.max(), freq=RESAMPLE)
data = data.reindex(grid)                                  # grille régulière : les trous deviennent NaN
valid = data.notna().all(axis=1).values                    # minutes où tous les canaux ont une mesure
T = len(data)
print(f"\nminutes valides : {valid.sum()} / {T} ({100 * valid.mean():.1f} %)")

agg = data[MAINS].fillna(0).values.astype(np.float32)
Yw = data[APPLIANCES].fillna(0).values.astype(np.float32).T     # (n_app, T) en W
thr = np.array([ON_W[a] for a in APPLIANCES], dtype=np.float32)

i_test = int(T * (1 - TEST_FRAC)); i_val = int(T * (1 - TEST_FRAC - VAL_FRAC))
for name, a, b in [("entraînement", 0, i_val), ("validation", i_val, i_test), ("test", i_test, T)]:
    print(f"{name:13s}: {data.index[a]} -> {data.index[b - 1]}  ({valid[a:b].sum()} minutes valides)")
tr = valid.copy(); tr[i_val:] = False                       # minutes valides d'entraînement
te = valid.copy(); te[:i_test] = False                      # minutes valides de test
if tr.sum() < 1000 or te.sum() < 1000:
    raise SystemExit("Pas assez de données valides : vérifie LOAD et les fichiers envoyés.")

## 3. Préparation : consommation de fond, échelles, entrées du réseau
- **Consommation de fond** : médiane de (total − somme des appareils) sur l'entraînement (≈ 178 W). Elle est utilisée par le CO.
- **Échelle de chaque appareil** : sa puissance « haute » (99e centile quand il est allumé), pour que le réseau travaille avec des valeurs de l'ordre de 1.
- **Cibles** : la puissance normalisée de chaque appareil et son état allumé (puissance > `ON_W`).
- **Entrées du réseau** : la puissance totale (en kW) et l'heure de la journée codée en sinus et cosinus.

In [ ]:
offset = float(np.median(np.clip(agg[tr] - Yw[:, tr].sum(0), 0, None)))
print(f"Consommation de fond estimée : {offset:.0f} W")
print(f"Part de l'énergie venant des {n_app} appareils suivis : {Yw[:, tr].sum() / agg[tr].sum() * 100:.0f} %")

scale_y = np.array([np.percentile(Yw[i, tr & (Yw[i] > thr[i])], 99) if (tr & (Yw[i] > thr[i])).any() else 1.0
                    for i in range(n_app)], dtype=np.float32)
print("échelles (W) :", dict(zip(APPLIANCES, scale_y.round(0).tolist())))

rows = [agg / 1000.0]
if USE_TIME:
    hour = (data.index.hour + data.index.minute / 60).values
    rows += [np.sin(2 * np.pi * hour / 24), np.cos(2 * np.pi * hour / 24)]
X = np.vstack(rows).astype(np.float32)
X[:, ~valid] = 0.0                                          # entrées neutres dans les trous
Y = (Yw / scale_y[:, None]).astype(np.float32)
ON = (Yw > thr[:, None]).astype(np.float32)
M = valid.astype(np.float32)
prev = ON[:, tr].mean(1)
print("part du temps allumé (entraînement) :", dict(zip(APPLIANCES, prev.round(3).tolist())))

## 4. Les deux références : naïf et CO (comme en semaine 2)
Même découpe que le réseau : le CO apprend ses niveaux de puissance sur la période d'entraînement seulement, puis teste les 64 combinaisons d'états à chaque minute du test.

In [ ]:
def kmeans1d(x, k, iters=100):
    if len(x) > 50000:
        x = np.random.default_rng(0).choice(x, 50000, replace=False)
    c = np.quantile(x, np.linspace(0.1, 0.9, k)) if k > 1 else np.array([x.mean()])
    for _ in range(iters):
        lab = np.abs(x[:, None] - c[None, :]).argmin(1)
        new = np.array([x[lab == j].mean() if (lab == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    return np.sort(c)

states = {}
for i, a in enumerate(APPLIANCES):
    on = Yw[i][tr & (Yw[i] > thr[i])]
    if len(on) < 10:
        states[a] = np.array([0.0]); continue
    k = min(N_ON_STATES[a], max(1, len(np.unique(on.round()))))
    states[a] = np.concatenate([[0.0], kmeans1d(on, k)])

combo_states = np.array(list(product(*[states[a] for a in APPLIANCES])))
combo_sums = combo_states.sum(axis=1)
idx_te = np.where(te)[0]
y_in = np.clip(agg[idx_te] - offset, 0, None)
best = np.empty(len(idx_te), dtype=int)
for i in range(0, len(idx_te), 20000):
    best[i:i + 20000] = np.abs(y_in[i:i + 20000, None] - combo_sums[None, :]).argmin(1)
pred_co = np.zeros((n_app, T), dtype=np.float32)
pred_co[:, idx_te] = combo_states[best].T
on_co = pred_co > thr[:, None]

mean_on = np.array([Yw[i, tr & (Yw[i] > thr[i])].mean() if (tr & (Yw[i] > thr[i])).any() else 0.0 for i in range(n_app)], dtype=np.float32)
pred_naive = np.repeat(mean_on[:, None], T, axis=1)
on_naive = np.ones((n_app, T), dtype=bool)
print("références prêtes :", len(combo_sums), "combinaisons pour le CO")

## 5. Le réseau
Un réseau **convolutif 1D dilaté à connexions résiduelles** :
- convolution d'entrée 1×1 (3 → 64 canaux) ;
- **8 blocs** de convolution de noyau 3, de dilatations 1, 2, 4… 128 minutes, chacun suivi d'un bloc **Squeeze-and-Excitation** (qui pondère les canaux selon leur utilité sur la fenêtre) et d'un dropout de 10 % ;
- convolution de sortie 1×1 (64 → 8 canaux) : 4 puissances et 4 scores « allumé ».

Le champ réceptif est d'environ ±255 minutes. Cette variante compte environ **116 000 paramètres** (≈ 450 Ko en float32), soit plus de 5 fois la version de base.

**Perte** : *smooth L1* sur la puissance (passée par une *softplus* pour rester positive) + **focal loss** sur l'état allumé, qui donne plus de poids aux minutes difficiles, avec un poids renforcé (jusqu'à ×20) pour les appareils rarement allumés. Les morceaux d'entraînement sont tirés en priorité parmi ceux où au moins un appareil est allumé.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation léger : pondère chaque canal."""
    def __init__(self, ch, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(ch, ch // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(ch // reduction, ch, bias=False),
            nn.Sigmoid(),
        )
    def forward(self, x):
        # x: (B, C, T)
        w = x.mean(dim=2)                 # (B, C)
        w = self.fc(w).unsqueeze(2)       # (B, C, 1)
        return x * w

class Net(nn.Module):
    def __init__(self, n_in, n_out, ch=64, dils=(1, 2, 4, 8, 16, 32, 64, 128)):
        super().__init__()
        self.inp = nn.Conv1d(n_in, ch, 1)
        self.blocks = nn.ModuleList()
        self.se = nn.ModuleList()
        for d in dils:
            self.blocks.append(nn.Conv1d(ch, ch, 3, padding=d, dilation=d))
            self.se.append(SEBlock(ch))
        self.out = nn.Conv1d(ch, n_out, 1)
        # Dropout spatial pour régulariser
        self.dropout = nn.Dropout(0.1)
    def forward(self, x):                       # x : (lot, n_in, temps) -> (lot, n_out, temps)
        h = F.relu(self.inp(x))
        for b, se in zip(self.blocks, self.se):
            h = h + se(F.relu(b(h)))
            h = self.dropout(h)
        return self.out(h)

net = Net(X.shape[0], 2 * n_app, CH).to(device)
n_par = sum(p.numel() for p in net.parameters())
print(f"{n_par} paramètres  ≈ {n_par * 4 / 1024:.0f} Ko en float32, ≈ {n_par / 1024:.0f} Ko en int8 (cible STM32)")

Xt, Yt, ONt = (torch.tensor(a, device=device) for a in (X, Y, ON))
Mt = torch.tensor(M, device=device)
pos_w = torch.tensor(np.clip((1 - prev) / np.maximum(prev, 1e-4), 1, 20), dtype=torch.float32, device=device).view(-1, 1)

def focal_loss(logits, targets, gamma=2.0, pos_weight=None):
    """Focal loss pour le déséquilibre ON/OFF."""
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1 - p) * (1 - targets)
    loss = bce * ((1 - p_t) ** gamma)
    return loss

def loss_fn(out, y, on, m):
    # out : (lot, 2*n_app, temps) ; y, on : (lot, n_app, temps) ; m : (lot, 1, temps)
    reg, logit = out[:, :n_app], out[:, n_app:]
    denom = m.sum() * n_app + 1e-6
    l_reg = (F.smooth_l1_loss(F.softplus(reg), y, reduction="none") * m).sum() / denom
    if USE_FOCAL:
        l_cls = (focal_loss(logit, on, gamma=FOCAL_GAMMA, pos_weight=pos_w) * m).sum() / denom
    else:
        l_cls = (F.binary_cross_entropy_with_logits(logit, on, pos_weight=pos_w, reduction="none") * m).sum() / denom
    return l_reg + l_cls

def get_batch(lo, hi):
    # Sampling stratifié : essayer de prendre des fenêtres avec des appareils allumés
    starts = []
    attempts = 0
    while len(starts) < BATCH and attempts < BATCH * 10:
        s = np.random.randint(lo, hi - CROP)
        # Vérifier qu'il y a au moins un appareil allumé dans la fenêtre
        if ONt[:, s:s+CROP].sum() > 0:
            starts.append(s)
        attempts += 1
    # Remplir le reste si nécessaire
    while len(starts) < BATCH:
        starts.append(np.random.randint(lo, hi - CROP))
    starts = np.array(starts)
    idx = torch.tensor(starts[:, None] + np.arange(CROP)[None, :], device=device)    # (lot, temps)
    x = Xt[:, idx].permute(1, 0, 2)
    y = Yt[:, idx].permute(1, 0, 2)
    on = ONt[:, idx].permute(1, 0, 2)
    m = Mt[idx].unsqueeze(1)
    return x, y, on, m

@torch.no_grad()
def run(a, b, ctx=512):
    """Sortie du réseau sur [a, b), avec ctx minutes de contexte avant a."""
    lo = max(0, a - ctx)
    out = net(Xt[:, lo:b].unsqueeze(0))[0]
    return out[:, a - lo:]

def val_loss():
    net.eval()
    out = run(i_val, i_test).unsqueeze(0)
    l = loss_fn(out, Yt[:, i_val:i_test].unsqueeze(0), ONt[:, i_val:i_test].unsqueeze(0), Mt[i_val:i_test].view(1, 1, -1)).item()
    net.train()
    return l

## 6. Entraînement
À chaque époque, je tire 100 lots de 32 morceaux de 12 h dans la période d'entraînement. L'optimiseur est **AdamW** (Adam avec régularisation des poids), avec écrêtage du gradient pour stabiliser l'apprentissage et réduction du pas d'apprentissage quand la validation stagne. La **validation** (la tranche juste avant le test) sert à arrêter quand le réseau cesse de progresser ; je garde les poids de la meilleure époque, sauvegardés dans `nilm_cnn_optimized.pt`.

La courbe d'apprentissage permet de repérer un éventuel sur-apprentissage (la validation qui remonte pendant que l'entraînement baisse).

In [ ]:
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=SCHEDULER_FACTOR, patience=SCHEDULER_PATIENCE, verbose=True)

best_val, best_state, bad, hist = 1e9, None, 0, []
t0 = time.time()
for ep in range(1, EPOCHS + 1):
    net.train(); tot = 0.0
    for _ in range(STEPS):
        x, y, on, m = get_batch(0, i_val)
        loss = loss_fn(net(x), y, on, m)
        opt.zero_grad(); loss.backward()
        # Gradient clipping pour stabilité
        torch.nn.utils.clip_grad_norm_(net.parameters(), GRAD_CLIP)
        opt.step()
        tot += loss.item()
    vl = val_loss(); hist.append((tot / STEPS, vl))
    scheduler.step(vl)
    current_lr = opt.param_groups[0]["lr"]
    print(f"époque {ep:2d}  lr {current_lr:.2e}  perte entraînement {tot / STEPS:.4f}  validation {vl:.4f}  ({time.time() - t0:.0f} s)")
    if vl < best_val - 1e-4:
        best_val, bad = vl, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad += 1
        if bad >= PATIENCE:
            print("arrêt : plus de progrès sur la validation"); break
net.load_state_dict(best_state)
torch.save(best_state, "nilm_cnn_optimized.pt")
print("meilleure perte de validation :", round(best_val, 4), "| poids sauvegardés dans nilm_cnn_optimized.pt")

In [ ]:
h = np.array(hist)
fig, ax = plt.subplots()
ax.plot(h[:, 0], label="entraînement"); ax.plot(h[:, 1], label="validation")
ax.set_xlabel("époque"); ax.set_ylabel("perte"); ax.set_title("Courbe d'apprentissage (si la validation remonte : sur-apprentissage)")
ax.legend(); plt.show()

## 7. Prédiction sur le test
Je fais passer toute la période de test dans le réseau (avec 512 minutes de contexte avant). La probabilité d'être allumé est une sigmoïde du score ; l'appareil est déclaré allumé au-dessus de 0,5, et sa puissance (remise en watts) est mise à 0 sinon.

In [ ]:
net.eval()
out = run(i_test, T).cpu().numpy()
reg = F.softplus(torch.tensor(out[:n_app])).numpy() * scale_y[:, None]
p_on = 1 / (1 + np.exp(-out[n_app:]))
on_nn = np.zeros((n_app, T), dtype=bool); on_nn[:, i_test:] = p_on > 0.5
pred_nn = np.zeros((n_app, T), dtype=np.float32); pred_nn[:, i_test:] = reg * (p_on > 0.5)
print("prédiction terminée sur", te.sum(), "minutes de test valides")

## 8. Comparaison : le tableau à retenir
Mêmes métriques qu'en semaine 2 (F1, précision, rappel, MAE, erreur d'énergie), **uniquement sur les minutes valides de la période de test**. Pour chaque appareil, la comparaison qui compte est **Réseau contre CO** ; le second tableau donne directement le gain de F1 et les erreurs d'énergie des deux méthodes.

In [ ]:
dt_h = pd.Timedelta(RESAMPLE).total_seconds() / 3600

def metrics(y_true, y_pred, on_pred, thr_a):
    t, p = y_true > thr_a, on_pred
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    e_t, e_p = y_true.sum() * dt_h / 1000, y_pred.sum() * dt_h / 1000
    return {
        "F1": 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else np.nan,
        "précision": tp / (tp + fp) if (tp + fp) else np.nan,
        "rappel": tp / (tp + fn) if (tp + fn) else np.nan,
        "MAE (W)": float(np.abs(y_true - y_pred).mean()),
        "kWh réels": e_t, "kWh prédits": e_p,
        "erreur énergie (%)": 100 * abs(e_p - e_t) / e_t if e_t > 0 else np.nan,
    }

rows = []
for method, pred, on in [("Naïf", pred_naive, on_naive), ("CO", pred_co, on_co), ("Réseau", pred_nn, on_nn)]:
    for i, a in enumerate(APPLIANCES):
        rows.append({"appareil": a, "méthode": method, **metrics(Yw[i, te], pred[i, te], on[i, te], thr[i])})
results = pd.DataFrame(rows).set_index(["appareil", "méthode"]).sort_index()
results.round(2)

In [ ]:
f1 = results["F1"].unstack("méthode")[["Naïf", "CO", "Réseau"]]
ax = f1.plot.bar(rot=0, color=["#B0BEC5", "#D99200", "#0E7C86"], title="F1 par appareil (période de test)", ylim=(0, 1))
ax.set_ylabel("F1 (1 = parfait)"); plt.show()

cmp = pd.DataFrame({"F1 CO": f1["CO"], "F1 réseau": f1["Réseau"], "gain F1": f1["Réseau"] - f1["CO"],
                    "erreur énergie CO (%)": results.xs("CO", level="méthode")["erreur énergie (%)"],
                    "erreur énergie réseau (%)": results.xs("Réseau", level="méthode")["erreur énergie (%)"]})
cmp.round(2)

## 9. Une journée de test : réel, CO, réseau
Sur la journée de test la plus active, je superpose pour chaque appareil la puissance réelle, celle du CO et celle du réseau, pour vérifier que les cycles sont trouvés au bon moment, avec la bonne durée et sans cycles inventés.

In [ ]:
idx_t = data.index[i_test:]
vt = valid[i_test:]
day_energy = pd.Series(Yw[:, i_test:].sum(0)[vt], index=idx_t[vt]).groupby(idx_t[vt].date).sum()
day = str(day_energy.idxmax())                         # jour de test avec le plus d'activité
print("jour affiché :", day)
sel = np.array([str(d.date()) == day for d in data.index])
t_ax = data.index[sel]
fig, axes = plt.subplots(n_app, 1, figsize=(11, 2.3 * n_app), sharex=True)
for ax, i, a in zip(np.atleast_1d(axes), range(n_app), APPLIANCES):
    ax.plot(t_ax, Yw[i, sel], color="#D99200", label="réel")
    ax.plot(t_ax, pred_co[i, sel], color="#8E9AAF", label="CO", alpha=0.8)
    ax.plot(t_ax, pred_nn[i, sel], color="#0E7C86", label="réseau", alpha=0.9)
    ax.set_title(a + " (W)", fontsize=10); ax.legend(loc="upper right", fontsize=8)
plt.tight_layout(); plt.show()

## 10. Résultats et analyse
**Version de base du réseau** (32 canaux, 7 blocs, 22 120 paramètres), sur la période de test :

| Appareil | F1 CO | F1 réseau | Erreur d'énergie CO | Erreur d'énergie réseau |
|---|---|---|---|---|
| Lave-vaisselle | 0,06 | **0,90** | 363 % | 15 % |
| Lave-linge | 0,23 | **0,88** | 123 % | 22 % |
| Bouilloire | 0,30 | **0,69** | 78 % | 28 % |
| Frigo | 0,50 | **0,80** | 23 % | 19 % |

Le réseau bat largement le CO sur les quatre appareils, surtout sur les appareils à cycles rares (lave-vaisselle, lave-linge) : il les reconnaît à leur forme dans le temps, là où le CO les confond avec la bouilloire. La règle du projet est donc remplie.

**Variante optimisée (ce notebook).** Elle n'a pas apporté de gain net par rapport à la version de base, pour plus de 5 fois plus de paramètres et un entraînement plus long. Avec seulement une maison pour apprendre, ajouter de la capacité n'aide pas : la limite vient des données, pas de la taille du réseau.

**Ce que je regarde dans les résultats :**
- un gain de F1 positif justifie le réseau pour l'appareil concerné ;
- l'erreur d'énergie est plus parlante que le F1 pour la flexibilité (c'est la quantité de kWh que l'on saura décaler) ;
- la courbe d'apprentissage indique si le réseau sur-apprend ;
- la taille du modèle conditionne son portage sur le microcontrôleur.

## 11. Conclusion
- **Modèle retenu** : le réseau convolutif dilaté **de base** (32 canaux, 7 blocs résiduels de dilatations 1 à 64 minutes, 22 120 paramètres, ≈ 22 Ko en int8). La variante élargie de ce notebook (≈ 116 000 paramètres) n'apporte pas de gain clair et serait plus coûteuse à embarquer : je ne la garde pas.
- **Entrées** : 3 valeurs par minute, la puissance totale de la maison en kW et l'heure de la journée (sinus, cosinus), sur une fenêtre de plusieurs heures.
- **Sorties** : 8 valeurs par minute, la puissance de chacun des 4 appareils (frigo, bouilloire, lave-linge, lave-vaisselle) et un score « allumé » pour chacun.
- **Performances (version de base)** : F1 de 0,69 à 0,90 contre 0,06 à 0,50 pour le CO, et une erreur d'énergie de 15 à 28 % contre 23 à 363 %.
- **Intérêt** : un seul capteur au tableau électrique suffit pour suivre les principaux appareils, avec une précision suffisante pour chiffrer et décaler leur consommation ; et le modèle est assez petit pour un microcontrôleur, ce qui permet de n'envoyer que des résultats sur le lien NB-IoT bas débit.
- **Limites** : une seule maison d'entraînement, donc un risque de mauvaise généralisation à une autre maison ; la bouilloire, très brève, reste l'appareil le plus difficile.
- **Suite** : semaine 4, analyse fine des erreurs (matrices de confusion) et première simulation de flexibilité (chauffe-eau, batterie).